# Track branch graphs — input/target visualization

This notebook is meant to inspect the H5 branch graphs produced by your converter.

It focuses on:
- **inputs**: node features, edges, geometry, buckets
- **targets**: `y_track = [pt_over_q, eta, phi]`
- **dataset-level checks**: graph sizes, target distributions, metadata consistency

The notebook assumes each H5 file contains:

- `graphs/<graph_id>/x`
- `graphs/<graph_id>/pos_m`
- `graphs/<graph_id>/dir_u`
- `graphs/<graph_id>/bucket`
- `graphs/<graph_id>/edge_index`
- `graphs/<graph_id>/edge_attr`
- `graphs/<graph_id>/y_track`

and graph attributes such as:
- `event_hash`
- `branch_idx`
- `majority_truth_id`
- `majority_count`
- `branch_size`
- `majority_pdgid`
- `charge`


## 1. Setup

Update `DATA_GLOB` so it points to the H5 parts produced by your script, for example:

```python
DATA_GLOB = "/path/to/data/track_graphs_pu0_part*.h5"
```

You can then run all cells top to bottom.


In [ ]:
from pathlib import Path
import glob
import math

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True


In [ ]:
# --- user config ---
DATA_GLOB = "./data/track_graphs_pu0_part*.h5"
MAX_INDEX_GRAPHS = None   # set e.g. 5000 for a fast partial scan
DEFAULT_GRAPH_IDX = 0


## 2. Discover files

In [ ]:
files = sorted(glob.glob(DATA_GLOB))
print(f"Found {len(files)} H5 file(s)")
for f in files[:10]:
    print(" -", f)
if not files:
    raise FileNotFoundError(
        f"No H5 files matched DATA_GLOB={DATA_GLOB!r}. "
        "Please update the path in the config cell."
    )


## 3. Build a lightweight graph index

In [ ]:
def build_graph_index(files, max_graphs=None):
    rows = []
    graph_counter = 0

    for file_idx, path in enumerate(files):
        with h5py.File(path, "r") as h5:
            graphs = h5["graphs"]
            graph_names = sorted(graphs.keys())

            for local_name in graph_names:
                g = graphs[local_name]

                y_track = np.asarray(g["y_track"], dtype=np.float32)
                n_nodes = int(g["x"].shape[0])
                n_edges = int(g["edge_index"].shape[1])

                rows.append(
                    {
                        "global_graph_idx": graph_counter,
                        "file_idx": file_idx,
                        "file_path": path,
                        "graph_name": local_name,
                        "event_hash": int(g.attrs.get("event_hash", -1)),
                        "branch_idx": int(g.attrs.get("branch_idx", -1)),
                        "majority_truth_id": int(g.attrs.get("majority_truth_id", -1)),
                        "majority_count": int(g.attrs.get("majority_count", -1)),
                        "branch_size_attr": int(g.attrs.get("branch_size", -1)),
                        "majority_pdgid": int(g.attrs.get("majority_pdgid", 0)),
                        "charge": float(g.attrs.get("charge", np.nan)),
                        "n_nodes": n_nodes,
                        "n_edges": n_edges,
                        "pt_over_q_GeV": float(y_track[0]) if len(y_track) > 0 else np.nan,
                        "eta": float(y_track[1]) if len(y_track) > 1 else np.nan,
                        "phi": float(y_track[2]) if len(y_track) > 2 else np.nan,
                    }
                )

                graph_counter += 1
                if max_graphs is not None and graph_counter >= max_graphs:
                    return pd.DataFrame(rows)

    return pd.DataFrame(rows)

index_df = build_graph_index(files, max_graphs=MAX_INDEX_GRAPHS)
print(f"Indexed {len(index_df)} graph(s)")
index_df.head()


## 4. Helper functions

In [ ]:
def load_graph_by_row(row):
    path = row["file_path"]
    graph_name = row["graph_name"]

    with h5py.File(path, "r") as h5:
        g = h5["graphs"][graph_name]

        data = {
            "file_path": path,
            "graph_name": graph_name,
            "attrs": {k: g.attrs[k] for k in g.attrs.keys()},
            "original_node_ids": np.asarray(g["original_node_ids"]),
            "x": np.asarray(g["x"], dtype=np.float32),
            "pos_m": np.asarray(g["pos_m"], dtype=np.float32),
            "dir_u": np.asarray(g["dir_u"], dtype=np.float32),
            "bucket": np.asarray(g["bucket"], dtype=np.int64),
            "edge_index": np.asarray(g["edge_index"], dtype=np.int64),
            "edge_attr": np.asarray(g["edge_attr"], dtype=np.float32),
            "y_track": np.asarray(g["y_track"], dtype=np.float32),
        }

    return data


def load_graph(global_graph_idx):
    row = index_df.iloc[int(global_graph_idx)]
    return load_graph_by_row(row)


def graph_summary_dict(graph):
    y = graph["y_track"]
    attrs = graph["attrs"]

    return {
        "file_path": graph["file_path"],
        "graph_name": graph["graph_name"],
        "event_hash": int(attrs.get("event_hash", -1)),
        "branch_idx": int(attrs.get("branch_idx", -1)),
        "majority_truth_id": int(attrs.get("majority_truth_id", -1)),
        "majority_count": int(attrs.get("majority_count", -1)),
        "branch_size_attr": int(attrs.get("branch_size", -1)),
        "majority_pdgid": int(attrs.get("majority_pdgid", 0)),
        "charge": float(attrs.get("charge", np.nan)),
        "n_nodes": int(graph["x"].shape[0]),
        "n_edges": int(graph["edge_index"].shape[1]),
        "x_shape": tuple(graph["x"].shape),
        "edge_attr_shape": tuple(graph["edge_attr"].shape),
        "target_pt_over_q_GeV": float(y[0]),
        "target_eta": float(y[1]),
        "target_phi": float(y[2]),
    }


def show_graph_summary(graph):
    summary = pd.Series(graph_summary_dict(graph), name="value")
    display(summary.to_frame())


def node_feature_dataframe(graph):
    x = graph["x"]
    pos = graph["pos_m"]
    dire = graph["dir_u"]
    bucket = graph["bucket"]

    cols = [
        "x_pos_m", "y_pos_m", "z_pos_m",
        "dir_x", "dir_y", "dir_z",
        "bucket_chamberIndex", "bucket_layers", "bucket_sector", "bucket_segments"
    ]

    df = pd.DataFrame(x, columns=cols)
    df["node_id"] = np.arange(len(df))
    df["original_node_id"] = graph["original_node_ids"]
    df["r_m"] = np.linalg.norm(pos[:, :2], axis=1)
    df["theta_deg"] = np.degrees(np.arccos(np.clip(dire[:, 2], -1.0, 1.0)))
    return df[["node_id", "original_node_id"] + cols + ["r_m", "theta_deg"]]


def edge_feature_dataframe(graph):
    ei = graph["edge_index"]
    ea = graph["edge_attr"]

    if ei.shape[1] == 0:
        return pd.DataFrame(columns=[
            "src", "dst",
            "dpos_x", "dpos_y", "dpos_z",
            "dist", "cosang", "same_chamber", "same_sector"
        ])

    df = pd.DataFrame(
        {
            "src": ei[0],
            "dst": ei[1],
            "dpos_x": ea[:, 0],
            "dpos_y": ea[:, 1],
            "dpos_z": ea[:, 2],
            "dist": ea[:, 3],
            "cosang": ea[:, 4],
            "same_chamber": ea[:, 5],
            "same_sector": ea[:, 6],
        }
    )
    return df


## 5. Dataset-level overview

In [ ]:
summary_df = pd.DataFrame(
    {
        "n_graphs": [len(index_df)],
        "n_files": [index_df["file_path"].nunique()],
        "mean_nodes": [index_df["n_nodes"].mean()],
        "median_nodes": [index_df["n_nodes"].median()],
        "mean_edges": [index_df["n_edges"].mean()],
        "median_edges": [index_df["n_edges"].median()],
        "pt_over_q_mean_GeV": [index_df["pt_over_q_GeV"].mean()],
        "eta_mean": [index_df["eta"].mean()],
        "phi_mean": [index_df["phi"].mean()],
    }
)
summary_df


In [ ]:
fig = plt.figure()
plt.hist(index_df["n_nodes"], bins=40)
plt.xlabel("nodes per graph")
plt.ylabel("count")
plt.title("Graph size distribution")
plt.show()

fig = plt.figure()
plt.hist(index_df["n_edges"], bins=40)
plt.xlabel("edges per graph")
plt.ylabel("count")
plt.title("Edge count distribution")
plt.show()

fig = plt.figure()
plt.hist(index_df["pt_over_q_GeV"], bins=60)
plt.xlabel("target pt/q [GeV]")
plt.ylabel("count")
plt.title("Target pt/q distribution")
plt.show()

fig = plt.figure()
plt.hist(index_df["eta"], bins=60)
plt.xlabel("target eta")
plt.ylabel("count")
plt.title("Target eta distribution")
plt.show()

fig = plt.figure()
plt.hist(index_df["phi"], bins=60)
plt.xlabel("target phi")
plt.ylabel("count")
plt.title("Target phi distribution")
plt.show()


In [ ]:
fig = plt.figure()
plt.scatter(index_df["eta"], index_df["phi"], s=8)
plt.xlabel("eta")
plt.ylabel("phi")
plt.title("Target eta-phi coverage")
plt.show()

fig = plt.figure()
plt.scatter(index_df["pt_over_q_GeV"], index_df["eta"], s=8)
plt.xlabel("pt/q [GeV]")
plt.ylabel("eta")
plt.title("Target pt/q vs eta")
plt.show()


## 6. Load one graph

In [ ]:
graph_idx = DEFAULT_GRAPH_IDX
graph = load_graph(graph_idx)
show_graph_summary(graph)


In [ ]:
node_df = node_feature_dataframe(graph)
edge_df = edge_feature_dataframe(graph)

print("Node table:")
display(node_df.head(20))

print("Edge table:")
display(edge_df.head(20))


## 7. Visualize the graph input

In [ ]:
def plot_graph_3d(graph, color_by="bucket_sector", show_direction=False, max_arrows=100):
    pos = graph["pos_m"]
    dirs = graph["dir_u"]
    bucket = graph["bucket"]
    ei = graph["edge_index"]

    color_map = {
        "bucket_sector": bucket[:, 2],
        "bucket_chamber": bucket[:, 0],
        "bucket_layer": bucket[:, 1],
        "bucket_segments": bucket[:, 3],
        "z": pos[:, 2],
        "r": np.linalg.norm(pos[:, :2], axis=1),
    }
    c = color_map.get(color_by, bucket[:, 2])

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    for k in range(ei.shape[1]):
        i, j = int(ei[0, k]), int(ei[1, k])
        ax.plot(
            [pos[i, 0], pos[j, 0]],
            [pos[i, 1], pos[j, 1]],
            [pos[i, 2], pos[j, 2]],
            alpha=0.25,
        )

    sc = ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=c, s=40)

    if show_direction:
        n = min(len(pos), max_arrows)
        ax.quiver(
            pos[:n, 0], pos[:n, 1], pos[:n, 2],
            dirs[:n, 0], dirs[:n, 1], dirs[:n, 2],
            length=0.25, normalize=True
        )

    for i in range(len(pos)):
        ax.text(pos[i, 0], pos[i, 1], pos[i, 2], str(i), fontsize=8)

    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_zlabel("z [m]")
    ax.set_title(f"3D branch graph (color_by={color_by})")
    fig.colorbar(sc, ax=ax, shrink=0.7)
    plt.show()


def plot_graph_projections(graph, color_by="bucket_sector"):
    pos = graph["pos_m"]
    bucket = graph["bucket"]
    ei = graph["edge_index"]

    color_map = {
        "bucket_sector": bucket[:, 2],
        "bucket_chamber": bucket[:, 0],
        "bucket_layer": bucket[:, 1],
        "bucket_segments": bucket[:, 3],
        "z": pos[:, 2],
        "r": np.linalg.norm(pos[:, :2], axis=1),
    }
    c = color_map.get(color_by, bucket[:, 2])

    r = np.linalg.norm(pos[:, :2], axis=1)

    fig = plt.figure(figsize=(8, 6))
    for k in range(ei.shape[1]):
        i, j = int(ei[0, k]), int(ei[1, k])
        plt.plot([pos[i, 2], pos[j, 2]], [r[i], r[j]], alpha=0.25)
    plt.scatter(pos[:, 2], r, c=c, s=50)
    for i in range(len(pos)):
        plt.text(pos[i, 2], r[i], str(i), fontsize=8)
    plt.xlabel("z [m]")
    plt.ylabel("r = sqrt(x^2+y^2) [m]")
    plt.title(f"z-r projection (color_by={color_by})")
    plt.show()

    fig = plt.figure(figsize=(8, 6))
    for k in range(ei.shape[1]):
        i, j = int(ei[0, k]), int(ei[1, k])
        plt.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]], alpha=0.25)
    plt.scatter(pos[:, 0], pos[:, 1], c=c, s=50)
    for i in range(len(pos)):
        plt.text(pos[i, 0], pos[i, 1], str(i), fontsize=8)
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.title(f"x-y projection (color_by={color_by})")
    plt.axis("equal")
    plt.show()


In [ ]:
plot_graph_3d(graph, color_by="bucket_sector", show_direction=True)
plot_graph_projections(graph, color_by="bucket_sector")


## 8. Inspect the target output for the selected graph

In [ ]:
y = graph["y_track"]
target_df = pd.DataFrame(
    {
        "target_name": ["pt_over_q_GeV", "eta", "phi"],
        "value": y,
    }
)
target_df


In [ ]:
fig = plt.figure()
plt.bar(["pt/q [GeV]", "eta", "phi"], y)
plt.title("Target values for selected graph")
plt.show()


## 9. Quick consistency checks

In [ ]:
def check_graph_consistency(graph):
    problems = []

    x = graph["x"]
    pos = graph["pos_m"]
    dire = graph["dir_u"]
    bucket = graph["bucket"]
    ei = graph["edge_index"]
    ea = graph["edge_attr"]
    attrs = graph["attrs"]

    if x.shape[1] != 10:
        problems.append(f"x has shape {x.shape}, expected second dim = 10")

    if pos.shape[1] != 3:
        problems.append(f"pos_m has shape {pos.shape}, expected (*, 3)")

    if dire.shape[1] != 3:
        problems.append(f"dir_u has shape {dire.shape}, expected (*, 3)")

    if bucket.shape[1] != 4:
        problems.append(f"bucket has shape {bucket.shape}, expected (*, 4)")

    if ei.shape[0] != 2:
        problems.append(f"edge_index has shape {ei.shape}, expected (2, E)")

    if ea.shape[1] != 7 and ea.shape[0] > 0:
        problems.append(f"edge_attr has shape {ea.shape}, expected (*, 7)")

    if x.shape[0] != pos.shape[0] or x.shape[0] != dire.shape[0] or x.shape[0] != bucket.shape[0]:
        problems.append("node-level arrays do not have the same number of rows")

    if ei.shape[1] != ea.shape[0]:
        problems.append("edge_index and edge_attr have inconsistent edge counts")

    if "branch_size" in attrs and int(attrs["branch_size"]) != x.shape[0]:
        problems.append(
            f"attr branch_size={int(attrs['branch_size'])} differs from x.shape[0]={x.shape[0]}"
        )

    norms = np.linalg.norm(dire, axis=1)
    if not np.allclose(norms, 1.0, atol=1e-3):
        problems.append("dir_u vectors are not all unit-normalized within tolerance")

    return problems

problems = check_graph_consistency(graph)
if problems:
    print("Consistency issues:")
    for p in problems:
        print(" -", p)
else:
    print("No obvious consistency issues found.")


## 10. Browse graphs by simple criteria

In [ ]:
# Examples:
# index_df.query("n_nodes >= 8 and abs(pt_over_q_GeV) > 20").head(20)
# index_df.sort_values("n_edges", ascending=False).head(20)
# index_df[index_df["majority_pdgid"] == 13].head(20)

index_df.head(20)


In [ ]:
# Pick a graph by filtering the index, then display it.
candidate_df = index_df.sort_values(["n_nodes", "n_edges"], ascending=False).head(20)
display(candidate_df)

graph_idx = int(candidate_df.iloc[0]["global_graph_idx"])
graph = load_graph(graph_idx)
show_graph_summary(graph)
plot_graph_3d(graph, color_by="bucket_sector", show_direction=False)
plot_graph_projections(graph, color_by="bucket_sector")


## 11. Optional: compare several graphs quickly

This gives a compact visual sense of how the target values vary across examples.


In [ ]:
def plot_target_cloud(sample_df):
    fig = plt.figure()
    plt.scatter(sample_df["eta"], sample_df["phi"], s=np.clip(sample_df["n_nodes"] * 5, 10, 100))
    plt.xlabel("eta")
    plt.ylabel("phi")
    plt.title("Target eta-phi (marker size ~ n_nodes)")
    plt.show()

    fig = plt.figure()
    plt.scatter(sample_df["pt_over_q_GeV"], sample_df["phi"], s=np.clip(sample_df["n_edges"] * 2, 10, 100))
    plt.xlabel("pt/q [GeV]")
    plt.ylabel("phi")
    plt.title("Target pt/q vs phi (marker size ~ n_edges)")
    plt.show()

sample_df = index_df.sample(min(len(index_df), 500), random_state=7) if len(index_df) else index_df
plot_target_cloud(sample_df)


## Notes

- `x` already contains the concatenated node input features:
  - `pos_m` (3)
  - `dir_u` (3)
  - `bucket` cast to float (4)

- `y_track` is the regression target:
  - `y_track[0] = pt/q` in **GeV**
  - `y_track[1] = eta`
  - `y_track[2] = phi`

- The notebook visualizes the **surviving branch subgraphs** written by the converter, not the full pre-threshold event graph.

- If you also want, I can prepare a second notebook that opens the original ROOT files and compares:
  1. the raw event segments,
  2. the candidate graph before ONNX thresholding,
  3. the final branch graph written to H5.
